Group F aka Humongous data Project 1 notebook

In [1]:
# Spark initialisation
from pyspark.context import SparkContext
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

sc = SparkContext('local', 'Project1_notebook')

spark = (
    SparkSession.builder
    .appName('Project1_notebook')
    .enableHiveSupport()   # persist tables to local Hive metastore
    .getOrCreate()
)

In [25]:
from pathlib import Path
INBOX_PATH = Path("data/inbox")
OUTBOX_PATH = Path("data/outbox")
MANIFEST_PATH = Path("state/manifest.json")

In [26]:
# Check input directory and read in required files based on what is new / changed since last time
import os
import json
import pyarrow.parquet as pq

# Load manifest
if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)
else:
    manifest = {"processed_files": []}

processed_files = {f["filename"]: f for f in manifest["processed_files"]}
file_paths_to_process = []

# Process files in inbox
for file_path in Path("data/inbox").glob("*_tripdata_*"):
    file_name = str(file_path)
    
    if file_name in processed_files:
        if processed_files[file_name]["size"] == file_path.stat().st_size:
            if  processed_files[file_name]["row_count"] ==  pq.ParquetFile(file_path).metadata.num_rows:
                print(f"Skipping {file_name}, already processed.")
                continue
    
    print("Processing file: " + str(file_path))
    
    if file_name not in processed_files:
        processed_files[file_name] = {"filename": file_name}
    processed_files[file_name]["status"] = "processing"
    file_paths_to_process.append(file_path)

manifest["processed_files"] = list(processed_files.values())
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

Processing file: data/inbox/yellow_tripdata_2025-01.parquet
Processing file: data/inbox/yellow_tripdata_2025-02.parquet


In [12]:
# Ingestion TODO 
# filter useful columns



In [27]:
# Transformations
# Otto

# trip_duration veergu lisades, märkasin, et osad sõidavad mitu päeva, kuigi läbitud vahemaa väike - ilmselt tuleks transfor-
# mationite käigus eemaldada? - Karen

In [28]:
# TESTIMISE JAOKS HETKEL HARDCODETUD. IDK, MIS LOOGIKA SIIA TULEB
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

if file_paths_to_process: 
    clean_df = spark.read.parquet(*[str(f) for f in file_paths_to_process])
else:
    print("No new files to process.")
    clean_df = spark.createDataFrame([], schema=None)  


In [29]:
# Enrichment
# Karen

# logic for creating required derived columns: trip_duration_minutes, pickup_date

clean_df = (clean_df
    .withColumn("trip_duration_minutes", 
        ((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60).cast("int")
    )
    .withColumn("pickup_date", F.date_format("tpep_pickup_datetime", "MMMM dd, yyyy"))
)


In [30]:
# kontrollpäring
clean_df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pickup_date",
    "trip_duration_minutes",
    "passenger_count",
    "trip_distance"
).show(10)

+--------------------+---------------------+-----------------+---------------------+---------------+-------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|      pickup_date|trip_duration_minutes|passenger_count|trip_distance|
+--------------------+---------------------+-----------------+---------------------+---------------+-------------+
| 2025-02-01 00:12:18|  2025-02-01 00:32:33|February 01, 2025|                   20|              3|         3.12|
| 2025-02-01 00:40:04|  2025-02-01 00:49:15|February 01, 2025|                    9|              1|          1.4|
| 2025-02-01 00:06:09|  2025-02-01 00:11:51|February 01, 2025|                    5|              0|          0.4|
| 2025-02-01 00:15:13|  2025-02-01 00:20:19|February 01, 2025|                    5|              0|          0.7|
| 2025-02-01 00:02:52|  2025-02-01 00:20:25|February 01, 2025|                   17|              1|         4.19|
| 2025-02-01 00:33:47|  2025-02-01 00:41:49|February 01, 2025|                  

In [31]:
# vaata üle, ei taha hardcodeda !!
zones_df = spark.read.parquet("data/inbox/taxi_zone_lookup.parquet")
zones_df.select("LocationID", "Zone").show()

+----------+--------------------+
|LocationID|                Zone|
+----------+--------------------+
|         1|      Newark Airport|
|         2|         Jamaica Bay|
|         3|Allerton/Pelham G...|
|         4|       Alphabet City|
|         5|       Arden Heights|
|         6|Arrochar/Fort Wad...|
|         7|             Astoria|
|         8|        Astoria Park|
|         9|          Auburndale|
|        10|        Baisley Park|
|        11|          Bath Beach|
|        12|        Battery Park|
|        13|   Battery Park City|
|        14|           Bay Ridge|
|        15|Bay Terrace/Fort ...|
|        16|             Bayside|
|        17|             Bedford|
|        18|        Bedford Park|
|        19|           Bellerose|
|        20|             Belmont|
+----------+--------------------+
only showing top 20 rows


In [32]:
from pyspark.sql.functions import broadcast

# clean_df - ilmselt pärast transformatione on vaja muuta

enriched_df = clean_df \
    .join(broadcast(zones_df.selectExpr("LocationID as PULocationID", "Zone as pickup_zone")), "PULocationID", "left") \
    .join(broadcast(zones_df.selectExpr("LocationID as DOLocationID", "Zone as dropoff_zone")), "DOLocationID", "left")

enriched_df.show(5)

+------------+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+-----------------+--------------------+-------------------+
|DOLocationID|PULocationID|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|trip_duration_minutes|      pickup_date|         pickup_zone|       dropoff_zone|
+------------+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+

In [ ]:
#Output

In [ ]:
# mapping, filter, 